## Data Downloading
In the paper, BraTS 2020 dataset is used for training model only so we only download Training Data and apply futher modification in this dataset. Later, we use BraTS 2019 and 2018 as testing data so we will not adjust any from those.

In [ ]:
import numpy as np 
import pandas as pd 
from pathlib import Path
import os

DATA_PATH = Path("/kaggle/input/datasets/awsaf49/brats20-dataset-training-validation")
TRAIN_PATH = DATA_PATH / "BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData"

patient_dirs = sorted([d for d in TRAIN_PATH.iterdir() if d.is_dir()])
print(f"Total training cases: {len(patient_dirs)}")
print(f"Example patient: {patient_dirs[0].name}")

## EDA & Visualization
Each patient folder contains 5 files:

   *_flair.nii  - Fluid Attenuated Inversion Recovery
   
   *_t1.nii     - T1-weighted
   
   *_t1ce.nii   - T1 contrast-enhanced
   
   *_t2.nii     - T2-weighted
   
   *_seg.nii    - Segmentation mask (ground truth)

In [ ]:
modalities = ["flair","t1","t1ce","t2","seg"]

def get_patient_files(patient_dir):
    """
    return: dict of {modality: filepath} for a patient
    """
    files = {}
    for mod in modalities:
        matches = list(patient_dir.glob(f"*_{mod}.nii*"))
        if matches:
            files[mod] = matches[0]
    return files

sample_files = get_patient_files(patient_dirs[0])
for mod, path in sample_files.items():
    print(f"  {mod:.6s}: {path.name}")

In [ ]:
import nibabel as nib

def load_volume(filepath):
    """
    Load a NIfTI file
    return: numpy array
    """
    img = nib.load(str(filepath))
    return img.get_fdata()
    
patient = patient_dirs[0]
files = get_patient_files(patient)

flair = load_volume(files["flair"])
seg = load_volume(files["seg"])

print(f"Volume shape : {flair.shape}")    # (240, 240, 155)
print(f"Voxel dtype  : {flair.dtype}")
print(f"Value range  : [{flair.min():.1f}, {flair.max():.1f}]")

# Segmentation labels:
# 0 = background, 1 = necrotic/non-enhancing tumor core
# 2 = peritumoral edema, 4 = GD-enhancing tumor
print(f"Seg labels   : {np.unique(seg)}")

In [ ]:
records = []

for patient_dir in patient_dirs:
    files = get_patient_files(patient_dir)
    if "seg" not in files:
        continue
    seg = load_volume(files["seg"])
    labels, counts = np.unique(seg, return_counts=True)
    label_dict = dict(zip(labels.astype(int), counts))

    records.append({
        "patient": patient_dir.name,
        "total_voxels": seg.size,
        "bg_voxels":    label_dict.get(0, 0),
        "ncr_voxels":   label_dict.get(1, 0),    # necrotic core
        "edema_voxels": label_dict.get(2, 0),    # peritumoral edema
        "enhancing_voxels": label_dict.get(4, 0) # GD-enhancing
    })

df = pd.DataFrame(records)
df["tumor_voxels"] = df["ncr_voxels"] + df["edema_voxels"] + df["enhancing_voxels"]
df["tumor_fraction"] = df["tumor_voxels"] / df["total_voxels"]

print(df.head())

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(15,4))
for ax, col, label, color in zip(axes, ["ncr_voxels", "edema_voxels", "enhancing_voxels"],
                                       ["Necrotic Core", "Edema", "Enhancing Tumor"],
                                       ["steelblue", "orange", "crimson"]):
    ax.hist(df[col],bins=40,color=color,edgecolor="white")
    ax.set_title(label)
    ax.set_xlabel("Voxel count")
    
plt.tight_layout()
plt.show()

In [ ]:
def compute_intensity_stats(patient_dir,modality="flair"):
    files = get_patient_files(patient_dir)
    vol = load_volume(files[modality])
    mask = vol > 0
    brain = vol[mask]
    return {"mean" : brain.mean(), "std" : brain.std(),
            "p1" : np.percentile(brain,1), "p99" : np.percentile(brain,99)}

stats = [compute_intensity_stats(p,"flair") for p in patient_dirs[:50]]
stats_df = pd.DataFrame(stats)
print(stats_df.describe().round(2))

In [ ]:
stats_df.head()

## Data Preprocessing
Following the paper, the pipeline for preprocessing is simple and contain no augmentation because deep learning model is well-performed when trained with minimal processed image

Pipeline: Center Crop &rarr;  250 x 250 x 155 to 128 x 128 x 128 &rarr;  3-class masks

In [ ]:
OUTPUT_PATH = Path("/kaggle/working/preprocessed")
OUTPUT_PATH.mkdir(exist_ok=True)
TARGET_SHAPE = (128, 128, 128)

In [ ]:
def center_crop(vol, crop_shape):
    """
    Crop 3D volume to crop_shape from the center
    Pads with zseros if volume is smaller than crop_shape on any axis
    """
    res = np.zeros(crop_shape,dtype=vol.dtype)

    src_slices, dst_slices = [], []
    for dim in range(3):
        src_size = vol.shape[dim]
        tgt_size = crop_shape[dim]

        src_start = max((src_size - tgt_size) // 2, 0)
        src_end = src_start + min(tgt_size, src_size)

        dst_start = max((tgt_size - src_size) // 2, 0)
        dst_end = dst_start + (src_end - src_start)

        src_slices.append(slice(src_start, src_end))
        dst_slices.append(slice(dst_start, dst_end))

    res[tuple(dst_slices)] = vol[tuple(src_slices)]
    return res

In [ ]:
from scipy.ndimage import zoom

def reshape_volume(vol, tgt_shape, is_mask=False):
    """
    Reshape using zoom. Nearest-neighbor for masks, trilinear for images
    """
    factors = [t / s for t, s in zip(tgt_shape,vol.shape)]
    order = 0 if is_mask else 1
    return zoom(vol,factors,order=order).astype(vol.dtype)

In [ ]:
def normalize(vol):
    """
    z-score normalization on the brain region (non-zero voxels)
    Keeps background at 0
    """
    mask = vol > 0
    if mask.sum() == 0:
        return vol
    mean = vol[mask].mean()
    std = vol[mask].std()
    out = np.zeros_like(vol,dtype=np.float32)
    out[mask] = (vol[mask] - mean) / (std + 1e-8)
    return out

In [ ]:
def seg_to_multiclass(seg):
    """
    return: (3, H, W, D) binary array [WT, TC, ET]
    """
    wt = (seg > 0).astype(np.uint8)     #Eveything non-BG
    tc = ((seg == 1) | (seg == 4).astype(np.uint8))
    et = (seg == 4).astype(np.uint8)
    return np.stack([wt, tc, et],axis=0)


In [ ]:
def augment_intensity(image, shift_range=(-0.1, 0.1), scale_range=(0.9, 1.1)):
    """
    Apply per-channel random intensity augmentation on brain voxels only.
    
    Args:
        image : (4, 128, 128, 128) float32 — z-score normalized
        shift_range : additive shift sampled uniformly per channel
        scale_range : multiplicative scale sampled uniformly per channel
    Returns:
        augmented image (4, 128, 128, 128) float32
    """
    augmented = image.copy()
    
    for ch in range(image.shape[0]):
        brain_mask = augmented[ch] > augmented[ch].min()  # non-background voxels
        
        shift = np.random.uniform(*shift_range)
        scale = np.random.uniform(*scale_range)
        
        augmented[ch][brain_mask] = augmented[ch][brain_mask] * scale + shift
    
    return augmented

In [ ]:
crop_shape = (192, 192, 144)
Mods = ["flair", "t1", "t1ce", "t2"]
def preprocess_patient(patient_dir, save=True, augment=False):
    files = get_patient_files(patient_dir)
    pid   = patient_dir.name

    processed_imgs = {}
    for mod in Mods:
        vol = nib.load(str(files[mod])).get_fdata(dtype=np.float32)
        vol = center_crop(vol, crop_shape)        # 240×240×155 → 192×192×144
        vol = reshape_volume(vol, TARGET_SHAPE)    # → 128×128×128
        vol = normalize(vol)                      # z-score
        processed_imgs[mod] = vol

    image = np.stack([processed_imgs[m] for m in Mods], axis=0)  # (4,128,128,128)

    if augment:
        image = augment_intensity(image)

    seg = nib.load(str(files["seg"])).get_fdata(dtype=np.float32)
    seg = center_crop(seg, crop_shape)
    seg = reshape_volume(seg, TARGET_SHAPE, is_mask=True)
    mask = seg_to_multiclass(seg)                 # (3,128,128,128)

    if save:
        out_dir = OUTPUT_PATH / pid
        out_dir.mkdir(exist_ok=True)
        np.save(out_dir / "image.npy", image)
        np.save(out_dir / "mask.npy",  mask)

    return image, mask

In [ ]:
from tqdm.notebook import tqdm

patient_dirs = sorted([d for d in TRAIN_PATH.iterdir() if d.is_dir()])

failed = []
for patient_dir in tqdm(patient_dirs, desc="Preprocessing"):
    try:
        preprocess_patient(patient_dir, save=True)
    except Exception as e:
        failed.append((patient_dir.name, str(e)))

print(f"\n✅ Done. {len(patient_dirs) - len(failed)}/{len(patient_dirs)} patients processed.")
if failed:
    print("❌ Failed:", failed)

In [ ]:
sample_pid = patient_dirs[0].name
img  = np.load(OUTPUT_PATH / sample_pid / "image.npy")
msk  = np.load(OUTPUT_PATH / sample_pid / "mask.npy")

print(f"Image shape : {img.shape}  dtype: {img.dtype}")   # (4, 128, 128, 128) float32
print(f"Mask  shape : {msk.shape}  dtype: {msk.dtype}")   # (3, 128, 128, 128) uint8
print(f"Image range : [{img.min():.2f}, {img.max():.2f}]")
print(f"WT voxels   : {msk[0].sum()}")
print(f"TC voxels   : {msk[1].sum()}")
print(f"ET voxels   : {msk[2].sum()}")

In [ ]:
from pathlib import Path
from typing import Tuple, Optional, Dict, Callable
import torch
from torch.utils.data import Dataset

class ThomasDataset(Dataset):
    """
    PyTorch Dataset for 3D Brain Tumor Segmentation — BraTS 2020.

    Preprocessed structure (per patient):
        <data_path>/
            BraTS20_Training_XXX/
                image.npy  — float32 (4, 128, 128, 128)  [FLAIR,T1,T1CE,T2]
                mask.npy   — float32 (3, 128, 128, 128)  [WT, TC, ET]

    Args:
        data_path  : Path to preprocessed root directory.
        transform  : Optional dict with keys 'image' and 'mask',
                     each a callable applied to the respective tensor.
        augment    : If True, apply per-channel intensity shift/scale at load time.
                     Shift in (−0.1, 0.1), Scale in (0.9, 1.1).
                     Should be True only for training split.
    """

    MODALITIES   = ["FLAIR", "T1", "T1CE", "T2"]
    TARGET_SHAPE = (4, 128, 128, 128)   # expected image shape
    MASK_SHAPE   = (3, 128, 128, 128)   # expected mask  shape  [WT, TC, ET]

    SHIFT_RANGE  = (-0.1, 0.1)
    SCALE_RANGE  = ( 0.9, 1.1)

    def __init__(self, data_path, transform=None, augment= False):
        self.data_path = Path(data_path)
        self.augment   = augment

        if not self.data_path.exists():
            raise FileNotFoundError(f"Data path not found: {self.data_path}")

        # Validate transform dict format
        if transform:
            if not isinstance(transform, dict) or \
               "image" not in transform or "mask" not in transform:
                raise TypeError(
                    "transform must be a dict with 'image' and 'mask' keys."
                )
        self.transform = transform

        # Collect valid patient directories
        self.patient_dirs = sorted([
            d for d in self.data_path.iterdir()
            if d.is_dir()
            and (d / "image.npy").exists()
            and (d / "mask.npy").exists()
        ])

        if len(self.patient_dirs) == 0:
            raise RuntimeError(
                f"No valid patient folders found in {self.data_path}. "
                "Each folder must contain image.npy and mask.npy."
            )
            
    def __len__(self) -> int:
        return len(self.patient_dirs)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Returns:
            image_tensor : float32 (4, 128, 128, 128)
            mask_tensor  : float32 (3, 128, 128, 128)
        """
        if idx < 0 or idx >= len(self):
            raise IndexError(f"Index {idx} out of range for dataset of size {len(self)}.")

        patient_dir = self.patient_dirs[idx]

        image = np.array(np.load(patient_dir / "image.npy", mmap_mode="r"), dtype=np.float32)
        mask  = np.array(np.load(patient_dir / "mask.npy",  mmap_mode="r"), dtype=np.float32)

        if self.augment:
            image = self._augment_intensity(image)

        image_tensor = torch.from_numpy(image)
        mask_tensor  = torch.from_numpy(mask)

        if self.transform:
            image_tensor = self.transform["image"](image_tensor)
            mask_tensor  = self.transform["mask"](mask_tensor)

        return image_tensor, mask_tensor

    # ── Augmentation ────────────────────────────────────────
    def _augment_intensity(self, image: np.ndarray) -> np.ndarray:
        """
        Per-channel random intensity shift and scale on brain voxels only.
        Shift ∈ (−0.1, 0.1), Scale ∈ (0.9, 1.1).
        Background (min-value voxels) is left unchanged.
        """
        augmented = image.copy()
        for ch in range(augmented.shape[0]):
            brain_mask = augmented[ch] > augmented[ch].min()
            shift = np.random.uniform(*self.SHIFT_RANGE)
            scale = np.random.uniform(*self.SCALE_RANGE)
            augmented[ch][brain_mask] = augmented[ch][brain_mask] * scale + shift
        return augmented

    def get_patient_id(self, idx: int) -> str:
        """
        Returns the patient folder name for a 0-based index.
        e.g. dataset.get_patient_id(0) → 'BraTS20_Training_001'
        """
        if not (0 <= idx < len(self)):
            raise IndexError(
                f"Index {idx} out of range. Valid range: 0 to {len(self) - 1}."
            )
        return self.patient_dirs[idx].name


In [ ]:
from torch.utils.data import DataLoader, random_split
from sklearn.model_selection import KFold

OUTPUT_PATH = "/kaggle/working/preprocessed"

# ── 80/20 Train/Test split ─────────────────────────────────────
full_dataset = ThomasDataset(OUTPUT_PATH, augment=False)

n       = len(full_dataset)
n_train = int(0.8 * n)
n_test  = n - n_train

generator = torch.Generator().manual_seed(42)
train_indices, test_indices = random_split(
    range(n), [n_train, n_test], generator=generator
)
train_indices = list(train_indices)
test_indices  = list(test_indices)

# Build train/test datasets
train_dataset = ThomasDataset(OUTPUT_PATH, augment=True)
train_dataset.patient_dirs = [full_dataset.patient_dirs[i] for i in train_indices]

test_dataset = ThomasDataset(OUTPUT_PATH, augment=False)
test_dataset.patient_dirs  = [full_dataset.patient_dirs[i] for i in test_indices]

print(f"Total   : {n}")
print(f"Train   : {len(train_dataset)}  ({len(train_dataset)/n*100:.0f}%)")
print(f"Test    : {len(test_dataset)}   ({len(test_dataset)/n*100:.0f}%)")

# Test DataLoader
test_loader = DataLoader(
    test_dataset, batch_size=1, shuffle=False,
    num_workers=2, pin_memory=True
)

In [ ]:
K = 5
kfold = KFold(n_splits=K, shuffle=True, random_state=42)
all_dirs = train_dataset.patient_dirs   # only train patients go into folds

for fold, (fold_train_idx, fold_val_idx) in enumerate(kfold.split(all_dirs)):
    print(f"  Fold {fold+1}/{K} | train={len(fold_train_idx)}  val={len(fold_val_idx)}")
    print(f"{'─'*40}")

    # Fold train
    fold_train = ThomasDataset(OUTPUT_PATH, augment=True)
    fold_train.patient_dirs = [all_dirs[i] for i in fold_train_idx]

    # Fold val (no augment — used only for monitoring during training)
    fold_val = ThomasDataset(OUTPUT_PATH, augment=False)
    fold_val.patient_dirs   = [all_dirs[i] for i in fold_val_idx]

    fold_train_loader = DataLoader(
        fold_train, batch_size=1, shuffle=True,
        num_workers=2, pin_memory=True
    )
    fold_val_loader = DataLoader(
        fold_val, batch_size=1, shuffle=False,
        num_workers=2, pin_memory=True
    )

    ##TRAINING PROGRESS

In [ ]:
!pip install monai

In [ ]:
import torch
import numpy as np
import nibabel as nib
from pathlib import Path
from monai.networks.nets import SwinUNETR
from monai.inferers import sliding_window_inference

# ==========================================
# 1. PATH CONFIGURATION
# ==========================================
PREPROCESSED_PATH = Path("/kaggle/working/preprocessed")
WEIGHTS_PATH = Path("/kaggle/input/datasets/nguynlmhuy/swinunetr-bestfold-weights/swinunetr_fold4_best.pth")
OUTPUT_DIR = Path("/kaggle/working/swinunetr_predictions")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ==========================================
# 2. MODEL INITIALIZATION
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Running on device: {device}")

model = SwinUNETR(
    in_channels=4,
    out_channels=3,
    feature_size=24,
    depths=(2, 4, 2, 2),
    num_heads=(3, 6, 12, 24),
    drop_rate=0.0,
    attn_drop_rate=0.0,
    dropout_path_rate=0.0,
    use_checkpoint=True,
).to(device)

checkpoint = torch.load(WEIGHTS_PATH, map_location=device)
state_dict = checkpoint.get('state_dict', checkpoint.get('model', checkpoint))
model.load_state_dict(state_dict, strict=True)
model.eval()
print("[*] Successfully loaded pre-trained weights")

# ==========================================
# 3. UTILITY FUNCTIONS
# ==========================================
def convert_to_brats_labels(pred_array):
    """
    Input: pred_array shape (3, H, W, D) - Sigmoid output probabilities.
    Nested hierarchical logic mapping:
    - Channel 0 (WT - Whole Tumor): Comprises 1, 2, 4 -> Initialized to label 2 (Edema)
    - Channel 1 (TC - Tumor Core): Comprises 1, 4     -> Overwritten to label 1 (Necrotic Core)
    - Channel 2 (ET - Enhancing Tumor): Exclusively 4 -> Overwritten to label 4 (Enhancing Tumor)
    """
    # Initialize background mask (Label = 0)
    final_mask = np.zeros(pred_array.shape[1:], dtype=np.uint8)
    
    # Extract binary masks via a 0.5 probability threshold
    wt_mask = pred_array[0] > 0.5
    tc_mask = pred_array[1] > 0.5
    et_mask = pred_array[2] > 0.5

    # Apply hierarchical structural assignments
    final_mask[wt_mask] = 2
    final_mask[tc_mask] = 1
    final_mask[et_mask] = 4
    
    return final_mask

# ==========================================
# 4. INFERENCE PIPELINE
# ==========================================
patient_dirs = sorted([d for d in PREPROCESSED_PATH.iterdir() if d.is_dir()])
print(f"[*] Commencing inference for {len(patient_dirs)} patients...")

with torch.no_grad():
    for p_dir in patient_dirs:
        patient_id = p_dir.name

        image_path = p_dir / "image.npy"
        if not image_path.exists():
            print(f"  [!] Skipping {patient_id} — 'image.npy' not found")
            continue

        # Load image array: expected shape (4, H, W, D)
        image = np.load(image_path)                                  
        val_inputs = torch.from_numpy(image).float()                 
        val_inputs = val_inputs.unsqueeze(0).to(device)              

        # Execute sliding window inference for memory-efficient 3D segmentation
        val_outputs = sliding_window_inference(
            val_inputs, roi_size=(128, 128, 128),
            sw_batch_size=4, predictor=model, overlap=0.5
        )
        
        # Apply sigmoid activation and convert to numpy format
        val_outputs = torch.sigmoid(val_outputs).squeeze(0).cpu().numpy()  

        # Convert to standardized BraTS categorical labels
        final_seg = convert_to_brats_labels(val_outputs)             

        # Manage output directories
        patient_out_dir = OUTPUT_DIR / patient_id
        patient_out_dir.mkdir(exist_ok=True)

        # Export segmentation mask as .npy
        np.save(patient_out_dir / f"{patient_id}_pred.npy", final_seg)

        # Export segmentation mask as .nii.gz (Required for PyRadiomics compatibility)
        seg_nifti = nib.Nifti1Image(final_seg, affine=np.eye(4))
        nib.save(seg_nifti, str(patient_out_dir / f"{patient_id}_pred.nii.gz"))

        print(f"  Processed {patient_id} — Unique labels extracted: {np.unique(final_seg)}")

print("\n[*] INFERENCE PIPELINE COMPLETED SUCCESSFULLY!")

In [ ]:
!pip install --upgrade setuptools wheel
!pip install git+https://github.com/AIM-Harvard/pyradiomics.git

In [ ]:
import pandas as pd
import numpy as np
import SimpleITK as sitk
from pathlib import Path
from radiomics import featureextractor
import logging
import warnings
warnings.filterwarnings('ignore')

logging.getLogger('radiomics').setLevel(logging.ERROR)

# ==========================================
# 1. PATH CONFIGURATION
# ==========================================
PREPROCESSED_PATH = Path("/kaggle/working/preprocessed")
OUTPUT_DIR = Path("/kaggle/working/swinunetr_predictions")
SURVIVAL_CSV = Path("/kaggle/input/datasets/awsaf49/brats20-dataset-training-validation/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/survival_info.csv")

# Rename output file to indicate it is ready for the ML Pipeline
CSV_OUTPUT = Path("/kaggle/working/brats_raw_ready.csv") 

# ==========================================
# 2. PYRADIOMICS CONFIGURATION
# ==========================================
settings = {
    'binWidth': 25,
    'interpolator': 'sitkBSpline',
    'resampledPixelSpacing': None,
    'normalize': True,
}
extractor = featureextractor.RadiomicsFeatureExtractor(**settings)
extractor.enableImageTypeByName('Original')
extractor.enableImageTypeByName('LoG', customArgs={'sigma': [2.0]})

def create_sitk_image(numpy_array):
    return sitk.GetImageFromArray(numpy_array.transpose(2, 1, 0))

def get_binary_mask(seg_array, labels):
    return np.isin(seg_array, labels).astype(np.uint8)

# ==========================================
# 3. FEATURE EXTRACTION
# ==========================================
patient_dirs = sorted([d for d in PREPROCESSED_PATH.iterdir() if d.is_dir()])
print(f"[*] Commencing PyRadiomics extraction for {len(patient_dirs)} patients...")

all_features = []

for p_dir in patient_dirs:
    patient_id = p_dir.name
    image_path = p_dir / "image.npy"
    mask_path  = OUTPUT_DIR / patient_id / f"{patient_id}_pred.npy"

    if not image_path.exists() or not mask_path.exists():
        print(f"  [!] Skipping {patient_id} — missing required files")
        continue

    try:
        image_np = np.load(image_path)  # (4, H, W, D)
        mask_np  = np.load(mask_path)   # (H, W, D)

        img_flair_sitk = create_sitk_image(image_np[0])
        img_t1ce_sitk  = create_sitk_image(image_np[2])

        regions = {
            'WT': get_binary_mask(mask_np, [1, 2, 4]),
            'TC': get_binary_mask(mask_np, [1, 4]),
            'ET': get_binary_mask(mask_np, [4]),
        }

        modalities = {'FLAIR': img_flair_sitk, 'T1ce': img_t1ce_sitk}
        patient_feat = {'BraTS20ID': patient_id}

        for region_name, mask_array in regions.items():
            patient_feat[f"volume_{region_name}"] = int(mask_array.sum())

            if mask_array.sum() < 50:
                continue

            mask_sitk = sitk.Cast(create_sitk_image(mask_array), sitk.sitkUInt8)

            for mod_name, img_sitk in modalities.items():
                try:
                    mask_sitk.CopyInformation(img_sitk)
                    result = extractor.execute(img_sitk, mask_sitk)
                    for key, val in result.items():
                        if not key.startswith('diagnostics'):
                            patient_feat[f"{mod_name}_{region_name}_{key}"] = float(val)
                except Exception as e:
                    print(f"  [!] Extraction failed for {patient_id} {mod_name}_{region_name}: {e}")

        all_features.append(patient_feat)
        print(f"  Processed {patient_id}: {len(patient_feat)} features extracted")

    except Exception as e:
        print(f"  Error processing {patient_id}: {e}")

# ==========================================
# 4. CLINICAL COMPUTATION, OHE & LABEL MERGING
# ==========================================
print("\n[*] Calculating clinical volume ratios...")
df_features = pd.DataFrame(all_features)

# Safely append clinical ratios
if 'volume_WT' in df_features.columns:
    if 'volume_TC' in df_features.columns: 
        df_features['Ratio_TC_WT'] = df_features['volume_TC'] / (df_features['volume_WT'] + 1e-5)
    if 'volume_ET' in df_features.columns: 
        df_features['Ratio_ET_WT'] = df_features['volume_ET'] / (df_features['volume_WT'] + 1e-5)

print(f"[*] Reading and processing survival data from: {SURVIVAL_CSV}")
df_survival = pd.read_csv(SURVIVAL_CSV)
df_survival.rename(columns={df_survival.columns[0]: 'BraTS20ID'}, inplace=True)

# Safely cast survival days to numeric, dropping invalid rows
df_survival['Survival_days'] = pd.to_numeric(df_survival['Survival_days'], errors='coerce')
df_survival = df_survival.dropna(subset=['Survival_days'])

# Cast Age to numeric
if 'Age' in df_survival.columns:
    df_survival['Age'] = pd.to_numeric(df_survival['Age'], errors='coerce')

# One-Hot Encoding for Extent_of_Resection (EOR)
if 'Extent_of_Resection' in df_survival.columns:
    df_survival['Extent_of_Resection'] = df_survival['Extent_of_Resection'].fillna('Unknown')
    # Generates columns such as EOR_GTR, EOR_STR, etc.
    df_survival = pd.get_dummies(df_survival, columns=['Extent_of_Resection'], prefix='EOR', dtype=int)

# Perform inner join to retain only patients with both valid features and survival labels
df_final = pd.merge(df_features, df_survival, on='BraTS20ID', how='inner')

df_final.to_csv(CSV_OUTPUT, index=False)
print(f"Final integrated dataset saved at: {CSV_OUTPUT}")
print(f"  -> Total patients remaining after merge: {df_final.shape[0]}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.base import BaseEstimator, TransformerMixin

# ==========================================
# 1. CUSTOM COLLINEARITY FILTER DEFINITION
# ==========================================
class CollinearityFilter(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.90): 
        self.threshold = threshold
        self.keep_indices_ = None

    def fit(self, X, y=None):
        corr_matrix = np.abs(np.corrcoef(X, rowvar=False))
        upper = np.triu(corr_matrix, k=1)
        to_drop = [i for i in range(X.shape[1]) if any(upper[:, i] > self.threshold)]
        self.keep_indices_ = [c for c in range(X.shape[1]) if c not in to_drop]
        return self

    def transform(self, X, y=None):
        return X[:, self.keep_indices_]

# ==========================================
# 2. DATA LOADING & TRAIN/TEST SPLIT
# ==========================================
print("--- 1. LOADING RAW DATA ---")
# Load the cleaned and One-Hot Encoded raw data from the PyRadiomics step
df = pd.read_csv("/kaggle/working/brats_raw_ready.csv")

y = df['Survival_days'].values
X_df = df.drop(columns=['BraTS20ID', 'Survival_days'])

print("--- 2. TRAIN/TEST SPLIT (Preventing Data Leakage) ---")
# Splitting must occur BEFORE any Imputation/Scaling/Selection to ensure unbiased evaluation
X_train, X_test, y_train, y_test = train_test_split(X_df, y, test_size=0.2, random_state=42)

# ==========================================
# 3. FEATURE ENGINEERING PIPELINE SETUP
# ==========================================
print("--- 3. EXECUTING FEATURE ENGINEERING PIPELINE ---")
# Separate clinical variables (Age, EOR) from radiomic features
clinical_features = ['Age'] + [col for col in X_df.columns if col.startswith('EOR_')]
radiomic_features = [col for col in X_df.columns if col not in clinical_features]

# Dedicated preprocessing pipeline for radiomic features
radiomics_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('collinear', CollinearityFilter(threshold=0.90)),
    ('scaler', RobustScaler()),
    ('selector', SelectKBest(score_func=f_regression, k=50))
])

# Combine pipelines: Radiomics undergo transformation, Clinical features pass through unaltered
preprocessor = ColumnTransformer(
    transformers=[
        ('rad', radiomics_pipeline, radiomic_features),
        ('clin', 'passthrough', clinical_features)
    ]
)

# FIT ONLY ON THE TRAINING SET (No model training occurs here)
preprocessor.fit(X_train, y_train)

# ==========================================
# 4. EXPORT ENGINEERED DATASETS
# ==========================================
print("\n--- 4. EXPORTING ENGINEERED DATA FOR DOWNSTREAM TRAINING ---")

# Retrieve the names of the features that survived both filters (Collinearity and SelectKBest)
rad_pipeline = preprocessor.named_transformers_['rad']
kept_collinear_idx = rad_pipeline.named_steps['collinear'].keep_indices_
rad_features_step1 = [radiomic_features[i] for i in kept_collinear_idx]

kbest_mask = rad_pipeline.named_steps['selector'].get_support()
final_rad_features = [rad_features_step1[i] for i, val in enumerate(kbest_mask) if val]

# Final list of columns (Top Radiomics + Clinical features)
final_feature_names = final_rad_features + clinical_features

# Transform the datasets to extract the engineered numpy arrays
X_train_engineered = preprocessor.transform(X_train)
X_test_engineered = preprocessor.transform(X_test)

# Construct Training DataFrame
df_train_export = pd.DataFrame(X_train_engineered, columns=final_feature_names)
df_train_export['Survival_days'] = y_train
df_train_export['BraTS20ID'] = df.loc[X_train.index, 'BraTS20ID'].values

# Construct Testing DataFrame
df_test_export = pd.DataFrame(X_test_engineered, columns=final_feature_names)
df_test_export['Survival_days'] = y_test
df_test_export['BraTS20ID'] = df.loc[X_test.index, 'BraTS20ID'].values

# Reorder columns for professional formatting (ID and Label first)
cols = ['BraTS20ID', 'Survival_days'] + final_feature_names
df_train_export = df_train_export[cols]
df_test_export = df_test_export[cols]

# Save to disk
TRAIN_CSV_PATH = "/kaggle/working/train_engineered_features.csv"
TEST_CSV_PATH = "/kaggle/working/test_engineered_features.csv"
df_train_export.to_csv(TRAIN_CSV_PATH, index=False)
df_test_export.to_csv(TEST_CSV_PATH, index=False)

print(f"✓ Training set saved (Shape: {df_train_export.shape}): {TRAIN_CSV_PATH}")
print(f"✓ Testing set saved (Shape: {df_test_export.shape}): {TEST_CSV_PATH}")
print("[*] PIPELINE COMPLETE: You now have two clean, engineered datasets ready for downstream modeling (e.g., XGBoost, ResDeepSurv).")

In [ ]:
!pip install lifelines

In [ ]:
!pip install scikit-survival

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sksurv.metrics import concordance_index_censored, integrated_brier_score
from xgboost import XGBRegressor
import joblib

# ==========================================
# UTILITY FUNCTION: BRESLOW BASELINE HAZARD FOR IBS
# ==========================================
def compute_breslow_baseline_hazard(train_risks, train_times, train_events):
    sort_idx = np.argsort(train_times)
    t, e, r = train_times[sort_idx], train_events[sort_idx], train_risks[sort_idx]
    
    clipped_r = np.clip(r, -20.0, 20.0)
    exp_r = np.exp(clipped_r)
    
    unique_times = np.unique(t[e == 1])
    if len(unique_times) == 0:
        return np.array([t.min()]), np.array([0.0])
        
    h0 = np.zeros_like(unique_times, dtype=float)
    for i, time in enumerate(unique_times):
        risk_set = t >= time
        denom = np.sum(exp_r[risk_set])
        h0[i] = np.sum(e[t == time]) / denom if denom > 0 else 0.0
        
    return unique_times, np.cumsum(h0)

def predict_survival_functions(test_risks, breslow_times, breslow_H0, eval_times):
    idx = np.searchsorted(breslow_times, eval_times, side="right") - 1
    H0_eval = np.where(idx >= 0, breslow_H0[idx], 0)
    clipped_risks = np.clip(test_risks, -20.0, 20.0)
    return np.exp(-H0_eval.reshape(1, -1) * np.exp(clipped_risks).reshape(-1, 1))

# ==========================================
# 1. LOAD ENGINEERED DATA
# ==========================================
print("--- 1. LOADING ENGINEERED DATA FROM PREVIOUS STAGE ---")
df_train = pd.read_csv("/kaggle/working/train_engineered_features.csv")
df_test = pd.read_csv("/kaggle/working/test_engineered_features.csv")

y_train_raw = df_train['Survival_days'].values
y_test_raw = df_test['Survival_days'].values

X_train = df_train.drop(columns=['BraTS20ID', 'Survival_days']).values
X_test = df_test.drop(columns=['BraTS20ID', 'Survival_days']).values

y_train_surv = np.array([(True, t) for t in y_train_raw], dtype=[('event', 'bool'), ('time', 'float')])
y_test_surv = np.array([(True, t) for t in y_test_raw], dtype=[('event', 'bool'), ('time', 'float')])

# ==========================================
# 2. 5-FOLD CROSS-VALIDATION & MODEL RETENTION
# ==========================================
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_c_indexes = []
fold_models = [] # List to retain all 5 trained models for the ensemble

print(f"\n--- 2. EXECUTING 5-FOLD CROSS-VALIDATION ON TRAINING SET ---")
for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y_train_raw[train_idx], y_train_raw[val_idx]
    y_val_surv = y_train_surv[val_idx]
    
    model_cv = XGBRegressor(
        objective='survival:cox', 
        n_estimators=400, 
        learning_rate=0.015,
        max_depth=4, 
        reg_lambda=20, 
        random_state=42
    )
    model_cv.fit(X_tr, y_tr)
    
    val_preds = model_cv.predict(X_val)
    fold_c_index = concordance_index_censored(y_val_surv['event'], y_val_surv['time'], val_preds)[0]
    
    cv_c_indexes.append(fold_c_index)
    fold_models.append(model_cv) # Archive model instance
    print(f"  Fold {fold+1}: Validation C-index = {fold_c_index:.4f}")

print(f"==> Mean Cross-Validation C-index: {np.mean(cv_c_indexes):.4f}")

# ==========================================
# 3. ENSEMBLE INFERENCE (TEST SET EVALUATION)
# ==========================================
print("\n--- 3. ENSEMBLE INFERENCE ON HELD-OUT TEST SET (UNSEEN DATA) ---")

ensemble_train_risks = np.mean([model.predict(X_train) for model in fold_models], axis=0)
ensemble_test_risks = np.mean([model.predict(X_test) for model in fold_models], axis=0)

c_index_test = concordance_index_censored(y_test_surv['event'], y_test_surv['time'], ensemble_test_risks)[0]

min_time = y_test_surv['time'].min()
max_time = min(y_train_surv['time'].max(), y_test_surv['time'].max()) - 1.0

if min_time < max_time:
    test_times = np.linspace(min_time, max_time, 100)
    
    # 1. Estimate baseline cumulative hazard from the training set (using the ensemble risks)
    b_times, b_H0 = compute_breslow_baseline_hazard(ensemble_train_risks, y_train_surv['time'], y_train_surv['event'])
    
    # 2. Derive survival probabilities for the test set across the specified time grid
    test_surv_probs = predict_survival_functions(ensemble_test_risks, b_times, b_H0, test_times)
    
    # 3. Compute IBS
    try:
        ibs_test = integrated_brier_score(y_train_surv, y_test_surv, test_surv_probs, test_times)
    except Exception as e:
        print(f"  [!] IBS Calculation Error: {e}")
        ibs_test = float('nan')
else:
    ibs_test = float('nan')

print(f"\n✓ FINAL ENSEMBLE PERFORMANCE ON TEST SET:")
print(f"  - C-index: {c_index_test:.4f}")
print(f"  - IBS:     {ibs_test:.4f}")

# ==========================================
# 4. MODEL EXPORT
# ==========================================
joblib.dump(fold_models, "/kaggle/working/brats_xgboost_ensemble_models.pkl")
print("\n✓ Successfully exported the 5-model ensemble object!")